# C7 可重复启动展示版

只上传本Notebook与同名ZIP一次。之后每次开机执行第1、2格；正常结束后，同一内核可再次执行第2格。**不要 Run All。**

名称仍为C7；四张专项图先原版后改进版，四个展示面板，没有nogt列表项。本版不改变算法，不加速原PL计算。首次启动前确认旧任务安全结束并保留记录；故障任务不能强行清场。


## 1．离线安装/校验（每次开机执行，可重复）

不访问板卡。已安装则核对原文件，不覆盖，不删记录。


In [ ]:
from pathlib import Path
import hashlib,json,zipfile,sys,uuid,shutil
home=Path.cwd().resolve()
archive=home/'C7_Demo_r1.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='2f8c1a21b542bce6eb2370d03610c45805cf510a559a3c6a5f73c06add26c053', 'DEMO_ZIP_SHA_MISMATCH'
runtime=home/'demo_runtime'
with zipfile.ZipFile(archive) as bundle:
    names=bundle.namelist()
    manifest=json.loads(bundle.read('demo_manifest.json'))
    assert len(names)==len(set(names)) and set(names)==set(manifest['files'])|{'demo_manifest.json'}
    for name in names:
        p=Path(name)
        assert not p.is_absolute() and '..' not in p.parts and ':' not in name and chr(92) not in name
        data=bundle.read(name)
        if name!='demo_manifest.json':
            ref=manifest['files'][name]
            assert len(data)==ref['bytes'] and hashlib.sha256(data).hexdigest()==ref['sha256'], name
    if not runtime.exists():
        assert shutil.disk_usage(home).free>=1024**3, 'SD_FREE_SPACE_BELOW_1_GIB'
        stage=home/('demo_install_'+uuid.uuid4().hex)
        stage.mkdir()
        for name in names:
            target=stage/name
            target.parent.mkdir(parents=True,exist_ok=True)
            with target.open('xb') as f:f.write(bundle.read(name))
        stage.rename(runtime)
    for name in names:
        assert (runtime/name).read_bytes()==bundle.read(name), 'EXISTING_INSTALL_CHANGED: '+name
loaded=sys.modules.get('c7_demo')
assert loaded is None or Path(loaded.__file__).resolve()==runtime/'c7_demo.py', 'DIFFERENT_DEMO_IN_KERNEL'
if str(runtime) not in sys.path:sys.path.insert(0,str(runtime))
import c7_demo
demo=c7_demo.open_demo(runtime)
print('C7_DEMO_OFFLINE_VERIFIED',runtime)
print('下一格经人工确认才启动板卡展示；未执行模型或FPGA操作。')


## 2．启动展示（人工确认板卡空闲）

把 AUTHORIZED 改为 True 后执行。正常运行时重复点这格只返回已有地址，不重复加载；正常结束后执行这格会新建会话。若发生异常，保留本内核与记录，不自动重试。端口占用时不强制结束其他程序。


In [ ]:
AUTHORIZED=False
assert AUTHORIZED, '请先确认旧任务正常结束、记录已保存、板卡空闲'
state=await demo.start(c7_demo.CONFIRMATION,port=8090)
print(state)
for url in state['urls']:print('打开展示网页：',url)


## 网页操作

打开上格打印的8090网页。选择 **im0004 / im0291 / im0319 / im0324**，配置 **C7、512×512、最终掩膜、112**。

1. 点 **① 运行原版 C7**，等待四图显示原版。
2. 点 **② 运行改进版 FPGA**，等待专项补标完成。
3. 使用 **查看原版 / 查看改进版** 对比；切换已完成版本不触发计算。切换另一张图后重复1、2。

保持Notebook内核与网页开启；不要在运行中重启内核或断电。下面第3格是**展示结束才执行**，不要刚启动就执行。


## 3．保存记录并正常结束（展示完才执行）

导出完成后正常关闭网页服务，保留所有记录。不复位板卡。忙碌或故障会拒绝关闭。正常结束后可再次执行第2格开始新一轮；下次开机只需执行第1、2格。


In [ ]:
from IPython.display import display,FileLink
saved=Path(await demo.finish())
print('C7_DEMO_CLOSED_IDLE；下次可再次启动。记录：',saved)
display(FileLink(str(saved.relative_to(Path.cwd()))))


## 4．可选：只导出，不关闭服务

平时不用执行。只有当前调用已经结束/返回异常后才可导出；仍在运行则会拒绝。不重启内核，不重跑故障请求。


In [ ]:
from IPython.display import display,FileLink
saved=Path(await demo.export())
display(FileLink(str(saved.relative_to(Path.cwd()))))
